# Notizen für Quantize

# Gyro

Zahlen, die quantisiert werden sollen:
- score(): Alle außer var355, Probas[] und Output[]
- infer(): Alles außer result


# Imports

In [934]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split

# Find largest float in C File

In [935]:
def find_minmax_float_in_c_file(filename):
    try:
        with open(filename, 'r') as file:
            content = file.read()
        
        # Regex für Gleitkommazahlen: +/- Zahlen mit Dezimalpunkt (kann führende und nachfolgende Ziffern oder nicht haben)
        pattern = re.compile(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?') # 
        
        # Finde alle Gleitkommazahlen
        floats = pattern.findall(content)
        
        # Konvertiere alle gefundenen Gleitkommazahlen in tatsächliche Floats
        float_numbers = [float(num) for num in floats]
        
        # Gibt die größte Zahl zurück, falls die Liste nicht leer ist
        if float_numbers:
            return [min(float_numbers), max(float_numbers)]
        else:
            return None

    except FileNotFoundError:
        print(f"Die Datei {filename} wurde nicht gefunden.")
        return None
    except Exception as e:
        print(f"Ein unerwarteter Fehler ist aufgetreten: {e}")
        return None


# Replace Floats in C File

Multiplikation mit 100.000.000, weil im Modell die größte Gleitkommazahl 
13.* und die kleinste Zahl -1.* ist.\
Datentyp Long kann [-2.147.483.648;2.147.483.647] abbilden \
2147483647 / 13,353132 = 160822468,241908 -> stark gerundet 100000000

In [936]:
def replace_floats_in_c_file(filename, datatype, multiplicator):
    supportedDatatypes = ['double', 'float', 'long long', 'long', 'int', 'char']

    if (datatype in supportedDatatypes) == False:
        raise Exception("Datatype is not supported!")
    
    try:
        with open(filename, 'r') as file:
            content = file.read()
        
        # Regex für Gleitkommazahlen
        float_pattern = re.compile(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?')

        # Funktion, um gefundene Gleitkommazahlen zu ersetzen
        def replace_function(match):
            float_str = match.group(0)
            float_number = float(float_str)
            modified_number = round(float_number * multiplicator)
            return str(modified_number)

        # Ersetze alle Gleitkommazahlen im Inhalt
        modified_content = float_pattern.sub(replace_function, content)

        modified_content = re.sub(r'\bdouble\b', datatype, modified_content, flags=re.IGNORECASE)


        # Schreibe die Änderungen zurück in die Datei oder in eine neue Datei
        with open(filename, 'w') as file:
            file.write(modified_content)
    
        print(f"Alle Gleitkommazahlen in '{filename}' wurden erfolgreich ersetzt.")

    except FileNotFoundError:
        print(f"Die Datei {filename} wurde nicht gefunden.")
    except Exception as e:
        print(f"Ein unerwarteter Fehler ist aufgetreten: {e}")


# Import Data

In [937]:
datasetsPath = '../datasets/gyro/' # Set location of dataset
gyro = pd.read_csv(datasetsPath + 'gyro_mobile.csv')

# Preprocessing dataframe
gyro = gyro.drop(columns='timestamp') # Drop timestamp column

xtrain, xtest, ytrain, ytest = train_test_split( # Split into training and test datasets
    gyro.iloc[:,:6],
    gyro.iloc[:,6:],
    test_size=0.2,
    random_state=0
)

# Determine Multiplicator

Multiplikator für Ganzzahl-Ops soll ermittelt werden\
Hierzu soll der (absolut) größte Wert aus dem Modell, sowie den Input-Daten ermittelt werden\
Der größtmögliche Wert des Zieldatentyps soll daraufhin durch diesen Wert geteilt werden\
Aus dem Ergebnis resultiert der Multiplikator

In [938]:
def getMultiplicator(modelfilepath, inputcsvpath, datatype: int, decpot: bool = False) -> int:
    
    modelmin = find_minmax_float_in_c_file(modelfilepath)[0]
    modelmax = find_minmax_float_in_c_file(modelfilepath)[1]
    
    inputfile = pd.read_csv(inputcsvpath, sep=';')
    dfmin = inputfile.to_numpy().min()
    dfmax = inputfile.to_numpy().max()

    minmin = min([dfmin,modelmin])
    if(minmin < 0):
        minmin = minmin * (-1)
    maxmax = max([dfmax,modelmax])

    absmax = max([minmin, maxmax])
    
    maxvaloftype = 0

    match datatype:
        case 0: # long long (signed)
            maxvaloftype = 223372036854775807
        case 1: # long (signed)
            maxvaloftype = 2147483647
        case 2: # int (signed)
            maxvaloftype = 32767
        case 3: # char (signed) 
            maxvaloftype = 127
        case _:
            raise Exception("Datatype is not supported!")

    multiplicator = int(maxvaloftype / absmax)

    if decpot:
        new_multiplicator = 1
        while multiplicator > 1:
            multiplicator = multiplicator / 10
            if multiplicator >= 1: new_multiplicator = new_multiplicator * 10
        return new_multiplicator

    return multiplicator


# Generate Code

In [939]:
def codeGen(datatype: str = 'double', genFile: bool = False):
    xtest = pd.read_csv('xtest_balanced.csv', sep=';')

    row_length = len(xtest.iloc[0])
    integers = ['long long','long','int','char']

    match datatype:
        case 'double': multiplicator = 1
        case 'float': multiplicator = 1
        case 'long long': multiplicator = getMultiplicator('gyro_double.ino', 'xtest_balanced.csv', 0, True)
        case 'long': multiplicator = getMultiplicator('gyro_double.ino', 'xtest_balanced.csv', 1, True)
        case 'int': multiplicator = getMultiplicator('gyro_double.ino', 'xtest_balanced.csv', 2, True)
        case 'char': multiplicator = getMultiplicator('gyro_double.ino', 'xtest_balanced.csv', 3, True)
        case _:
            raise Exception("Datatype is not supported!")
    
    # file creation
    if genFile:
        file = open('gencode.txt','w')
        file.write("void inputContainer(){\n")
        
        # Datatype is an integer
        if datatype in integers: 

            for i in range(len(xtest)):
                varname = "x" + str(i)
                
                # Printing the variable
                tempString = "\t" + str(datatype) + " " + str(varname) + "[] = {"
                for val in range(row_length-1):
                    value = int(round((xtest.iloc[i].iloc[val]*multiplicator),0))
                    tempString += str(value) + ","
                value = int(round((xtest.iloc[i].iloc[row_length-1] * multiplicator),0))
                tempString += str(value) + "};\n"
                file.write(tempString)
                tempString = "\tinfer(" + varname + ");\n"
                file.write(tempString)

            file.write("}\n") 

        # Datatype is a decimal
        else:  

            for i in range(len(xtest)):
                varname = "x" + str(i)
                
                # Printing the variable
                tempString = "\t" + str(datatype) + " " + str(varname) + "[] = {"
                for val in range(row_length-1):
                    tempString += str(xtest.iloc[i].iloc[val]) + ","
                tempString += str(xtest.iloc[i].iloc[row_length-1]) + "};\n"
                file.write(tempString)
                tempString = "\tinfer(" + varname + ");\n"
                file.write(tempString)

            file.write("}\n") 
    
    # NO file is created
    else:

        # Datatype is an integer
        if datatype in integers:

            for i in range(len(xtest)):
                varname = "x" + str(i)
                
                # Printing the variable
                print(f'{datatype} {varname}[] = {{',end="")
                for val in range(row_length-1):
                    value = round(xtest.iloc[i].iloc[val] * multiplicator)
                    print(value,end=',')
                value = int(round((xtest.iloc[i].iloc[row_length-1] * multiplicator),0))
                print(f'{value}}};')
                print(f'infer({varname});')

        # Datatype is a decimal
        else:

            for i in range(len(xtest)):
                varname = "x" + str(i)
                
                # Printing the variable
                print(f'{datatype} {varname}[] = {{',end="")
                for val in range(row_length-1):
                    print(xtest.iloc[i].iloc[val],end=',')
                print(f'{xtest.iloc[i].iloc[row_length-1]}}};')
                print(f'infer({varname});')
        
    return multiplicator
        

# Main


In [ ]:
# codeGen('char', True)


Exception: Datatype is not supported!

### Modify C-Files

In [ ]:
# filename = 'gyro_model_ino_quantized.c'
# filename = 'infer_quantized.c'

# max_float = find_largest_float_in_c_file(filename)
# min_float = find_largest_float_in_c_file(filename, retMin = True)
# print(f"Die größte Gleitkommazahl in der Datei ist: {max_float}")
# print(f"Die kleinste Gleitkommazahl in der Datei ist: {min_float}")

# replace_floats_in_c_file(filename)

### Create (1st) Compare-CSV

In [ ]:
# bc = pd.read_csv('baseCapture.csv', sep=';')
# nqf = pd.read_csv('no_quantized_float.csv', sep=';')
# nqd = pd.read_csv('no_quantized_double.csv', sep=';')

# df = pd.DataFrame({
#     "baseScore0": bc['baseScore_0'],
#     "inoScore0_float": nqf['inoScore0'],
#     "inoScore0_double": nqd['inoScore0'],
#     "baseScore1": bc['baseScore_1'],
#     "inoScore1_float": nqf['inoScore1'],
#     "inoScore1_double": nqd['inoScore1'],
#     "label": bc['label'],
#     "inoLabel_float": nqf['inoLabel'],
#     "inoLabel_double": nqd['inoLabel']
#     }
# )

# df = df.iloc[:250]

# df['inoLabel_float'] = df['inoLabel_float'].astype(int)
# df['inoLabel_double'] = df['inoLabel_double'].astype(int)

# df.to_csv('compare.csv', sep=',', index=False)